# Generating Biologically Inspired Graphs — Methods from the Literature

## Introduction: Moving Beyond Random Graph Models

In the previous notebooks, we explored foundational graph theory concepts and examined early random graph models (Erdős-Rényi, Watts-Strogatz, Barabási-Albert). These models were mathematically elegant but largely ignored a critical feature of real biological networks: **spatial organization**.

Real brain networks are embedded in physical space. Neurons are located in specific regions, and the cost of long-range connections is significantly higher than local ones. This spatial constraint fundamentally shapes network structure.

### The Central Question

**How do we generate networks that look like real brain networks?**

This notebook surveys the literature on methods designed to capture biological network properties. Each method makes different assumptions and captures different aspects of biological reality:

- Some emphasize **spatial embedding** (where nodes are placed matters)
- Some emphasize **heterogeneous weights** (connections have varying strengths)
- Some emphasize **modularity and hierarchy** (brain regions have internal structure)
- Some emphasize **dynamics** (networks evolve over time)

The key insight: **no single method captures everything**. Real biological networks likely arise from combining multiple mechanisms.

## Recap: Graph Construction Methods and Their Properties

Let's review the methods we've encountered, organized by what network properties they capture:

| Model | Spatial Position | Edge Weights | Heterogeneous Degree | Modularity | Efficiency |
|-------|:----------------:|:------------:|:-------------------:|:----------:|:----------:|
| **Erdős-Rényi (ER)** | ✗ | ✗ | ✗ | ✗ | Low (random, dense) |
| **Watts-Strogatz (WS)** | ✗ | ✗ | ✗ (uniform) | ✓ | High (small-world) |
| **Barabási-Albert (BA)** | ✗ | ✗ | ✓ (power-law) | ~ | Scale-free |
| **Spatial Random** | ✓ | ~ (optional) | ✗ | ✗ | Distance-dependent |
| **Inverse-square** | ✓ | ✓ | ~ | ✗ | Sparse, efficient |

### Key Observation

Real biological networks have:
- ✓ **Spatial organization** (nodes embedded in physical space)
- ✓ **Heterogeneous weights** (connections vary in strength)
- ✓ **Heterogeneous degree** (some nodes much more connected than others)
- ✓ **Modularity** (densely connected clusters with sparse between-cluster connections)
- ✓ **Hierarchical structure** (nested organization at multiple scales)
- ✓ **Resource efficiency** (sparse, preferring local connections)

Our task: find or design models that capture these properties.

In [ ]:
# Setup imports
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.spatial.distance import pdist, squareform
from scipy.stats import lognorm, ks_2samp
import pandas as pd
import seaborn as sns
from itertools import combinations

# Set style
plt.style.use('default')
sns.set_palette("husl")
np.random.seed(42)

print("Libraries imported successfully.")
print(f"NumPy version: {np.__version__}")
print(f"NetworkX version: {nx.__version__}")

## The Waxman Model (1988)

### Historical Context

The Waxman model, introduced by Bernard M. Waxman in 1988, was one of the first spatial network models designed to capture realistic network topology. Originally developed for modeling computer networks, it has become foundational in biological network research.

### Key Properties

**Connection probability** between nodes u and v:

$$P(u,v) = \beta \cdot \exp\left(-\frac{d(u,v)}{L \cdot \alpha}\right)$$

Where:
- **d(u,v)** = Euclidean distance between nodes u and v
- **L** = maximum distance in the network (diagonal of embedding space)
- **α** = parameter controlling the ratio of long-range to short-range edges (0 to 1)
  - α → 0: heavily favors local connections
  - α → 1: allows more long-range connections
- **β** = parameter controlling overall edge density (0 to 1)

### What Does It Capture?

✓ **Spatial embedding**: Connections depend on physical distance

✓ **Exponential decay**: Connection probability falls off smoothly with distance (more biologically realistic than step functions)

~ **Weights**: Original model is binary (connected or not), but can be extended

✗ **Modularity**: No explicit clustering or block structure

✗ **Hierarchy**: Connections are memoryless (no self-similar structure)

### Biological Relevance

The exponential decay of connection probability matches observations in neural systems where the cost of long-range connections scales with distance. However, real neural networks show more heterogeneity in degree and stronger clustering than the Waxman model typically produces.

In [ ]:
def generate_waxman_graph(n, alpha=0.4, beta=0.4, dimensions=2, seed=None):
    """
    Generate a Waxman random graph.
    
    Parameters
    ----------
    n : int
        Number of nodes
    alpha : float, default=0.4
        Parameter controlling long-range vs short-range connections (0 to 1)
    beta : float, default=0.4
        Parameter controlling overall edge density (0 to 1)
    dimensions : int, default=2
        Dimensionality of embedding space
    seed : int or None
        Random seed for reproducibility
    
    Returns
    -------
    G : networkx.Graph
        Graph with position attributes
    positions : dict
        Node positions {node_id: (x, y, ...)}
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Generate random node positions in unit hypercube
    positions = np.random.rand(n, dimensions)
    
    # Compute pairwise distances
    distances = squareform(pdist(positions, metric='euclidean'))
    
    # Maximum distance (diagonal of unit hypercube)
    L = np.sqrt(dimensions)
    
    # Connection probabilities
    P = beta * np.exp(-distances / (L * alpha))
    
    # Create graph
    G = nx.Graph()
    G.add_nodes_from(range(n))
    
    # Add edges based on probability
    for i in range(n):
        for j in range(i + 1, n):
            if np.random.rand() < P[i, j]:
                G.add_edge(i, j, weight=1.0, distance=distances[i, j])
    
    # Store positions as node attributes
    pos_dict = {i: positions[i] for i in range(n)}
    nx.set_node_attributes(G, {i: positions[i] for i in range(n)}, 'position')
    
    return G, pos_dict

# Generate example Waxman graphs with different parameters
G_waxman_sparse, pos_sparse = generate_waxman_graph(50, alpha=0.2, beta=0.3, seed=42)
G_waxman_dense, pos_dense = generate_waxman_graph(50, alpha=0.6, beta=0.5, seed=42)

print(f"Sparse Waxman graph (α=0.2, β=0.3):")
print(f"  Nodes: {G_waxman_sparse.number_of_nodes()}")
print(f"  Edges: {G_waxman_sparse.number_of_edges()}")
print(f"  Density: {nx.density(G_waxman_sparse):.4f}")
print(f"  Avg degree: {2 * G_waxman_sparse.number_of_edges() / G_waxman_sparse.number_of_nodes():.2f}")

print(f"\nDenser Waxman graph (α=0.6, β=0.5):")
print(f"  Nodes: {G_waxman_dense.number_of_nodes()}")
print(f"  Edges: {G_waxman_dense.number_of_edges()}")
print(f"  Density: {nx.density(G_waxman_dense):.4f}")
print(f"  Avg degree: {2 * G_waxman_dense.number_of_edges() / G_waxman_dense.number_of_nodes():.2f}")

In [ ]:
# Visualize Waxman graphs with different parameters
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sparse Waxman
ax = axes[0]
positions_2d_sparse = np.array([pos_sparse[i][:2] for i in range(len(pos_sparse))])
nx.draw_networkx_edges(G_waxman_sparse, pos_sparse, ax=ax, alpha=0.2, width=0.5)
nx.draw_networkx_nodes(G_waxman_sparse, pos_sparse, ax=ax, node_size=50, node_color='steelblue', alpha=0.7)
ax.set_title('Waxman Model: Sparse (α=0.2, β=0.3)\nFavors local connections', fontsize=12, fontweight='bold')
ax.axis('off')

# Dense Waxman
ax = axes[1]
positions_2d_dense = np.array([pos_dense[i][:2] for i in range(len(pos_dense))])
nx.draw_networkx_edges(G_waxman_dense, pos_dense, ax=ax, alpha=0.2, width=0.5)
nx.draw_networkx_nodes(G_waxman_dense, pos_dense, ax=ax, node_size=50, node_color='coral', alpha=0.7)
ax.set_title('Waxman Model: Denser (α=0.6, β=0.5)\nAllows more long-range connections', fontsize=12, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('waxman_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Waxman model visualization complete.")

## Hierarchical Network Models

### Background

Ravasz & Barabási (2003) proposed a class of hierarchical models that explicitly capture the modular, recursive structure observed in biological networks. The key insight: **real networks are often self-similar at different scales**.

### The Model

Start with a **seed network** (e.g., a fully connected module of M nodes). Then:

1. **Replicate** the seed module to create M copies
2. **Connect** one node from each copy to a central "hub" node
3. **Repeat** this process iteratively to create larger hierarchical structures

This construction naturally produces:

- **Modularity**: Dense within-module connections, sparse between modules
- **Scale-free degree distribution**: Hub nodes emerge naturally
- **High clustering**: Triangles are abundant in modules
- **Hierarchical organization**: Multiple levels of nested structure

### What Does It Capture?

✓ **Modularity**: Explicit community structure

✓ **Hierarchy**: Recursive, self-similar structure at multiple scales

✓ **Scale-free degree**: Power-law degree distributions emerge

✓ **High clustering**: Preserved through iterations

✗ **Spatial embedding**: Nodes have no inherent positions

~ **Weights**: Original model is unweighted; weight extensions exist but are not standard

### Biological Relevance

Real brains exhibit hierarchical organization across multiple scales: cortical columns, cortical areas, functional networks, whole-brain connectivity. This model captures that structure but ignores spatial constraints.

In [ ]:
def generate_hierarchical_modular_graph(num_modules=4, nodes_per_module=10, 
                                        p_intra=0.6, p_inter=0.05, seed=None):
    """
    Generate a hierarchical modular graph.
    
    Parameters
    ----------
    num_modules : int
        Number of modules
    nodes_per_module : int
        Nodes in each module
    p_intra : float
        Connection probability within modules
    p_inter : float
        Connection probability between modules
    seed : int or None
        Random seed
    
    Returns
    -------
    G : networkx.Graph
        Graph with 'module' node attribute
    """
    if seed is not None:
        np.random.seed(seed)
    
    G = nx.Graph()
    node_id = 0
    module_map = {}  # node -> module
    
    # Add nodes and intra-module edges
    for module in range(num_modules):
        module_nodes = []
        for _ in range(nodes_per_module):
            G.add_node(node_id)
            module_map[node_id] = module
            module_nodes.append(node_id)
            node_id += 1
        
        # Add intra-module edges
        for i, u in enumerate(module_nodes):
            for v in module_nodes[i+1:]:
                if np.random.rand() < p_intra:
                    G.add_edge(u, v, weight=1.0, type='intra')
    
    # Add inter-module edges
    all_nodes = list(G.nodes())
    for i, u in enumerate(all_nodes):
        for v in all_nodes[i+1:]:
            if module_map[u] != module_map[v]:
                if np.random.rand() < p_inter:
                    G.add_edge(u, v, weight=0.5, type='inter')
    
    # Store module assignments
    nx.set_node_attributes(G, module_map, 'module')
    
    return G, module_map

# Generate hierarchical modular graph
G_hier, modules = generate_hierarchical_modular_graph(num_modules=5, nodes_per_module=12, 
                                                        p_intra=0.5, p_inter=0.03, seed=42)

print(f"Hierarchical modular graph:")
print(f"  Nodes: {G_hier.number_of_nodes()}")
print(f"  Edges: {G_hier.number_of_edges()}")
print(f"  Density: {nx.density(G_hier):.4f}")
print(f"  Clustering coefficient: {nx.average_clustering(G_hier):.4f}")

# Count intra vs inter-module edges
intra_edges = sum(1 for u, v, d in G_hier.edges(data=True) if d.get('type') == 'intra')
inter_edges = G_hier.number_of_edges() - intra_edges
print(f"  Intra-module edges: {intra_edges}")
print(f"  Inter-module edges: {inter_edges}")

In [ ]:
# Visualize hierarchical modular graph
fig, ax = plt.subplots(figsize=(10, 8))

# Use spring layout for visualization
pos = nx.spring_layout(G_hier, k=0.5, iterations=50, seed=42)

# Color nodes by module
module_colors = [modules[node] for node in G_hier.nodes()]
cmap = plt.cm.tab10

# Draw edges
intra_edges = [(u, v) for u, v, d in G_hier.edges(data=True) if d.get('type') == 'intra']
inter_edges = [(u, v) for u, v, d in G_hier.edges(data=True) if d.get('type') == 'inter']

nx.draw_networkx_edges(G_hier, pos, edgelist=intra_edges, ax=ax, alpha=0.3, width=0.5, edge_color='gray')
nx.draw_networkx_edges(G_hier, pos, edgelist=inter_edges, ax=ax, alpha=0.1, width=0.5, 
                       edge_color='red', style='dashed')

# Draw nodes
nx.draw_networkx_nodes(G_hier, pos, node_color=module_colors, cmap=cmap, 
                       node_size=100, ax=ax, alpha=0.8, vmin=0, vmax=9)

ax.set_title('Hierarchical Modular Network\nDifferent colors = different modules', 
             fontsize=12, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('hierarchical_modular.png', dpi=100, bbox_inches='tight')
plt.show()

print("Hierarchical modular network visualization complete.")

## Stochastic Block Models

### Overview

Stochastic Block Models (SBMs) provide a flexible framework for generating networks with explicit community structure. Unlike the hierarchical model's specific construction, SBMs use a probabilistic framework:

1. **Assign** each node to a block (community/region) according to a prior distribution
2. **Draw edges** with probability depending on the blocks of the endpoints

### Model Definition

For each pair of nodes (u, v):
- If u and v are in the **same block**: P(edge) = p_in (high)
- If u and v are in **different blocks**: P(edge) = p_out (low)

### What Does It Capture?

✓ **Modularity**: Communities with specified internal/external densities

✓ **Flexibility**: Can have arbitrary number of blocks with different sizes

✓ **Generality**: Special cases include ER graphs and hierarchical models

~ **Hierarchy**: Can create multi-level blocks, but not as naturally as the hierarchical model

✗ **Spatial embedding**: No inherent spatial structure

✗ **Weights**: Standard model is binary (extensions exist)

### Biological Relevance

SBMs are particularly useful for modeling brain networks where blocks represent functional areas or anatomical regions. The model is flexible enough to accommodate known brain structure while remaining tractable for analysis.

In [ ]:
def generate_stochastic_block_model(block_sizes, p_matrix, seed=None):
    """
    Generate a stochastic block model.
    
    Parameters
    ----------
    block_sizes : list of int
        Size of each block
    p_matrix : 2D array-like
        Block-to-block connection probabilities
        p_matrix[i,j] = probability of edge between block i and block j
    seed : int or None
        Random seed
    
    Returns
    -------
    G : networkx.Graph
        Graph with 'block' node attribute
    """
    if seed is not None:
        np.random.seed(seed)
    
    p_matrix = np.asarray(p_matrix)
    num_blocks = len(block_sizes)
    
    # Create block assignment
    block_assignment = []
    node_id = 0
    for block, size in enumerate(block_sizes):
        block_assignment.extend([block] * size)
    
    total_nodes = sum(block_sizes)
    
    # Create graph
    G = nx.Graph()
    G.add_nodes_from(range(total_nodes))
    
    # Add edges
    for i in range(total_nodes):
        for j in range(i + 1, total_nodes):
            block_i = block_assignment[i]
            block_j = block_assignment[j]
            p = p_matrix[block_i, block_j]
            
            if np.random.rand() < p:
                G.add_edge(i, j, weight=1.0)
    
    # Store block assignments
    block_dict = {i: block_assignment[i] for i in range(total_nodes)}
    nx.set_node_attributes(G, block_dict, 'block')
    
    return G, block_dict

# Define block structure and connection probabilities
block_sizes = [15, 15, 15, 15]  # 4 blocks of 15 nodes each
p_matrix = np.array([
    [0.4, 0.05, 0.02, 0.02],  # Block 0: high intra, low inter
    [0.05, 0.4, 0.02, 0.02],  # Block 1: high intra, low inter
    [0.02, 0.02, 0.4, 0.05],  # Block 2: high intra, moderate to block 3
    [0.02, 0.02, 0.05, 0.4],  # Block 3: high intra, moderate to block 2
])

G_sbm, sbm_blocks = generate_stochastic_block_model(block_sizes, p_matrix, seed=42)

print(f"Stochastic Block Model:")
print(f"  Nodes: {G_sbm.number_of_nodes()}")
print(f"  Edges: {G_sbm.number_of_edges()}")
print(f"  Density: {nx.density(G_sbm):.4f}")
print(f"  Clustering coefficient: {nx.average_clustering(G_sbm):.4f}")

In [ ]:
# Visualize SBM
fig, ax = plt.subplots(figsize=(10, 8))

# Use spring layout
pos = nx.spring_layout(G_sbm, k=1.0, iterations=50, seed=42)

# Color nodes by block
block_colors = [sbm_blocks[node] for node in G_sbm.nodes()]
cmap = plt.cm.Set2

# Draw edges and nodes
nx.draw_networkx_edges(G_sbm, pos, ax=ax, alpha=0.2, width=0.5)
nx.draw_networkx_nodes(G_sbm, pos, node_color=block_colors, cmap=cmap, 
                       node_size=100, ax=ax, alpha=0.8, vmin=0, vmax=3)

ax.set_title('Stochastic Block Model\nClear modular structure with sparse between-block edges', 
             fontsize=12, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('stochastic_block_model.png', dpi=100, bbox_inches='tight')
plt.show()

print("Stochastic Block Model visualization complete.")

## The WREN Model: Weighted Random Evolving Networks

### Motivation

Most graph generation models treat edge weights as secondary or ignore them entirely. However, in biological networks:
- **Synaptic weights** vary by orders of magnitude
- **Strength distribution** is typically lognormal or power-law
- **Weights evolve** based on activity and learning

### Key Features

1. **Initial network**: Start with sparse random connections
2. **Weight reinforcement**: Frequently used connections strengthen
3. **Preferential attachment**: New connections more likely from high-strength nodes
4. **Distance dependence**: Local connections preferred (in spatial variant)

### The Model

At each time step:
- **Select** a random node u
- **Strengthen** edges from u (by small amount)
- **Potentially add** new edges from u based on distance and current weights
- **Occasional pruning**: Remove very weak edges

### What Does It Capture?

✓ **Heterogeneous weights**: Power-law or lognormal distributions emerge naturally

✓ **Preferential attachment**: Strong nodes get stronger

✓ **Rich-get-richer**: Natural inequality in network strength

~ **Spatial embedding**: Can be added as distance-dependent preferential attachment

~ **Hierarchy**: Emerges from temporal evolution

✗ **Explicit modularity**: Not built in, but may emerge from dynamics

### Biological Relevance

This model captures Hebbian learning dynamics where "neurons that fire together, wire together." The resulting weight distribution closely matches observations in neural systems.

In [ ]:
def generate_wren_graph(n, initial_edges=None, time_steps=100, 
                        reinforcement=0.1, spatial=False, seed=None):
    """
    Generate a Weighted Random Evolving Network.
    
    Parameters
    ----------
    n : int
        Number of nodes
    initial_edges : int or None
        Number of initial random edges. If None, use ~10% of possible edges
    time_steps : int
        Number of evolution steps
    reinforcement : float
        Weight increment per reinforcement event
    spatial : bool
        If True, embed nodes in 2D space and prefer local connections
    seed : int or None
        Random seed
    
    Returns
    -------
    G : networkx.Graph
        Weighted graph
    positions : dict or None
        Node positions if spatial=True
    """
    if seed is not None:
        np.random.seed(seed)
    
    G = nx.Graph()
    G.add_nodes_from(range(n))
    
    positions = None
    distances = None
    
    # Setup spatial embedding if requested
    if spatial:
        positions = np.random.rand(n, 2)
        distances = squareform(pdist(positions, metric='euclidean'))
    
    # Initialize with random edges
    if initial_edges is None:
        initial_edges = max(5, n // 10)
    
    for _ in range(initial_edges):
        u, v = np.random.choice(n, 2, replace=False)
        weight = np.random.exponential(0.1) + 0.1  # Initial weights from exponential distribution
        G.add_edge(u, v, weight=weight)
    
    # Evolution steps
    for step in range(time_steps):
        # Select random node
        u = np.random.randint(n)
        
        # Reinforce existing edges from u
        for v in G.neighbors(u):
            G[u][v]['weight'] += reinforcement
        
        # Potentially add new edge
        if np.random.rand() < 0.05:  # 5% chance per step
            # Find non-connected node with bias towards low distance (if spatial)
            non_neighbors = list(set(range(n)) - set(G.neighbors(u)) - {u})
            
            if non_neighbors:
                if spatial:
                    # Preferentially connect to nearby nodes
                    probs = 1.0 / (distances[u, non_neighbors] + 0.1)
                    probs /= probs.sum()
                    v = np.random.choice(non_neighbors, p=probs)
                else:
                    v = np.random.choice(non_neighbors)
                
                weight = np.random.exponential(0.1) + 0.05
                G.add_edge(u, v, weight=weight)
        
        # Occasional pruning of weak edges
        if step % 20 == 0:
            weak_edges = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] < 0.01]
            G.remove_edges_from(weak_edges)
    
    pos_dict = None
    if spatial and positions is not None:
        pos_dict = {i: positions[i] for i in range(n)}
        nx.set_node_attributes(G, pos_dict, 'position')
    
    return G, pos_dict

# Generate WREN graphs
G_wren_uniform, _ = generate_wren_graph(40, initial_edges=8, time_steps=200, 
                                         reinforcement=0.05, spatial=False, seed=42)
G_wren_spatial, pos_wren = generate_wren_graph(40, initial_edges=8, time_steps=200, 
                                                reinforcement=0.05, spatial=True, seed=42)

print(f"WREN graph (uniform space):")
print(f"  Nodes: {G_wren_uniform.number_of_nodes()}")
print(f"  Edges: {G_wren_uniform.number_of_edges()}")
weights_uniform = [d['weight'] for u, v, d in G_wren_uniform.edges(data=True)]
print(f"  Mean weight: {np.mean(weights_uniform):.4f}")
print(f"  Weight std: {np.std(weights_uniform):.4f}")
print(f"  Weight skewness: {np.mean(weights_uniform) / np.median(weights_uniform):.2f}x")

print(f"\nWREN graph (spatial):")
print(f"  Nodes: {G_wren_spatial.number_of_nodes()}")
print(f"  Edges: {G_wren_spatial.number_of_edges()}")
weights_spatial = [d['weight'] for u, v, d in G_wren_spatial.edges(data=True)]
print(f"  Mean weight: {np.mean(weights_spatial):.4f}")
print(f"  Weight std: {np.std(weights_spatial):.4f}")
print(f"  Weight skewness: {np.mean(weights_spatial) / np.median(weights_spatial):.2f}x")

In [ ]:
# Visualize WREN weight distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Weight distributions
ax = axes[0, 0]
ax.hist(weights_uniform, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
ax.set_xlabel('Edge weight', fontsize=10)
ax.set_ylabel('Frequency', fontsize=10)
ax.set_title('WREN (Uniform): Weight Distribution', fontsize=11, fontweight='bold')
ax.set_yscale('log')

ax = axes[0, 1]
ax.hist(weights_spatial, bins=20, alpha=0.7, color='coral', edgecolor='black')
ax.set_xlabel('Edge weight', fontsize=10)
ax.set_ylabel('Frequency', fontsize=10)
ax.set_title('WREN (Spatial): Weight Distribution', fontsize=11, fontweight='bold')
ax.set_yscale('log')

# Network visualization
ax = axes[1, 0]
pos_uniform = nx.spring_layout(G_wren_uniform, k=0.5, iterations=50, seed=42)
weights_norm = [d['weight'] / max(weights_uniform) * 2 for u, v, d in G_wren_uniform.edges(data=True)]
nx.draw_networkx_edges(G_wren_uniform, pos_uniform, width=weights_norm, alpha=0.5, ax=ax)
nx.draw_networkx_nodes(G_wren_uniform, pos_uniform, node_size=60, node_color='steelblue', 
                       alpha=0.7, ax=ax)
ax.set_title('WREN (Uniform): Network Structure', fontsize=11, fontweight='bold')
ax.axis('off')

ax = axes[1, 1]
if pos_wren:
    pos_list = [pos_wren[i] for i in range(len(pos_wren))]
    weights_norm_spatial = [d['weight'] / max(weights_spatial) * 2 for u, v, d in G_wren_spatial.edges(data=True)]
    nx.draw_networkx_edges(G_wren_spatial, pos_wren, width=weights_norm_spatial, alpha=0.5, ax=ax)
    nx.draw_networkx_nodes(G_wren_spatial, pos_wren, node_size=60, node_color='coral', 
                           alpha=0.7, ax=ax)
    ax.set_title('WREN (Spatial): Network Structure', fontsize=11, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('wren_models.png', dpi=100, bbox_inches='tight')
plt.show()

print("WREN visualization complete.")

## Distance-Dependent Weighted Graph Generation

### The Core Approach for Biologically Inspired Networks

The most promising approach for generating realistic biological networks combines:
1. **Spatial embedding** (nodes in physical space)
2. **Distance-dependent connection probability** (nearby nodes more likely to connect)
3. **Distance-dependent weights** (nearby connections stronger)

### Mathematical Foundation

For nodes u and v separated by distance d:

**Connection probability:**
$$P(\text{edge}) \propto \frac{1}{d^\alpha}$$

**Edge weight (if connected):**
$$w(u,v) \propto \frac{1}{d^\beta}$$

Typical values: α ≈ 2, β ≈ 1 (inverse-square law for probability, inverse for weight)

### Why This Works

1. **Natural sparsity**: Long-range connections exponentially rarer, avoiding complete graphs
2. **Efficient routing**: Short paths available due to high local density
3. **Small-world property**: Despite sparsity, graph diameter grows logarithmically
4. **Lognormal weights**: Automatic heavy-tailed weight distribution similar to real neurons
5. **Resource efficiency**: Low-cost local connections preferred

### Biological Evidence

- **C. elegans** connectome shows exponential decay with distance
- **Drosophila** larva brain: exponential/power-law distance dependence
- **Cortical columns**: Strong local clustering, weak long-range
- **Synaptic strengths**: Lognormal distribution across real neural systems

In [ ]:
def generate_bio_inspired_graph(n, dimensions=2, distance_exponent=2.0, 
                                weight_exponent=1.0, density_target=None, 
                                weight_function='inverse', seed=None):
    """
    Generate a biologically inspired weighted graph.
    
    Parameters
    ----------
    n : int
        Number of nodes
    dimensions : int
        Dimensionality of embedding space (2 or 3)
    distance_exponent : float
        Exponent for distance-dependent connection probability (typical: 2.0)
    weight_exponent : float
        Exponent for distance-dependent weights (typical: 1.0)
    density_target : float or None
        Target network density. If specified, scales connection probabilities to achieve it.
    weight_function : str
        'inverse': weight ~ 1/d^weight_exponent
        'exponential': weight ~ exp(-d * weight_exponent)
    seed : int or None
        Random seed
    
    Returns
    -------
    G : networkx.Graph
        Weighted graph with position attributes
    positions : dict
        Node positions
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Generate random node positions in unit hypercube
    positions = np.random.rand(n, dimensions)
    
    # Compute pairwise Euclidean distances
    distances = squareform(pdist(positions, metric='euclidean'))
    
    # Avoid division by zero
    np.fill_diagonal(distances, np.inf)
    
    # Distance-dependent connection probabilities
    # Add small offset to avoid singularity
    P = 1.0 / (distances ** distance_exponent + 1e-10)
    
    # Normalize probabilities
    if density_target is not None:
        # Scale to achieve target density
        current_expected_edges = np.sum(P) / 2  # Symmetric, so divide by 2
        target_edges = density_target * n * (n - 1) / 2
        scale_factor = target_edges / current_expected_edges if current_expected_edges > 0 else 1.0
        P = P * scale_factor
        P = np.clip(P, 0, 1)  # Keep probabilities valid
    
    # Create graph
    G = nx.Graph()
    G.add_nodes_from(range(n))
    
    # Add edges with distance-dependent weights
    for i in range(n):
        for j in range(i + 1, n):
            if np.random.rand() < P[i, j]:
                # Distance-dependent weight
                if weight_function == 'inverse':
                    weight = 1.0 / (distances[i, j] ** weight_exponent)
                elif weight_function == 'exponential':
                    weight = np.exp(-distances[i, j] * weight_exponent)
                else:
                    weight = 1.0
                
                G.add_edge(i, j, weight=weight, distance=distances[i, j])
    
    # Store positions
    pos_dict = {i: positions[i] for i in range(n)}
    nx.set_node_attributes(G, pos_dict, 'position')
    
    return G, pos_dict

# Generate biologically inspired graphs with different parameters
G_bio_1, pos_1 = generate_bio_inspired_graph(60, dimensions=2, distance_exponent=1.5, 
                                              weight_exponent=1.0, density_target=0.08, seed=42)
G_bio_2, pos_2 = generate_bio_inspired_graph(60, dimensions=2, distance_exponent=2.0, 
                                              weight_exponent=1.0, density_target=0.08, seed=42)
G_bio_3, pos_3 = generate_bio_inspired_graph(60, dimensions=2, distance_exponent=2.5, 
                                              weight_exponent=1.0, density_target=0.08, seed=42)

print("Biologically inspired graphs generated:")
for i, (G, exp) in enumerate([(G_bio_1, 1.5), (G_bio_2, 2.0), (G_bio_3, 2.5)], 1):
    weights = [d['weight'] for u, v, d in G.edges(data=True)]
    print(f"\nGraph {i} (distance_exponent={exp}):")
    print(f"  Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    print(f"  Density: {nx.density(G):.4f}")
    print(f"  Avg degree: {2*G.number_of_edges()/G.number_of_nodes():.2f}")
    print(f"  Weight range: [{min(weights):.4f}, {max(weights):.4f}]")
    print(f"  Weight median: {np.median(weights):.4f}")
    print(f"  Weight mean: {np.mean(weights):.4f}")

In [ ]:
# Visualize biologically inspired graphs
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

graphs = [(G_bio_1, pos_1, 1.5), (G_bio_2, pos_2, 2.0), (G_bio_3, pos_3, 2.5)]

for col, (G, pos, exp) in enumerate(graphs):
    # Top row: spatial layout
    ax = axes[0, col]
    
    # Get edge weights for visualization
    weights = [d['weight'] for u, v, d in G.edges(data=True)]
    weight_range = (min(weights), max(weights))
    weight_norm = [(d['weight'] - weight_range[0]) / (weight_range[1] - weight_range[0]) 
                   for u, v, d in G.edges(data=True)]
    
    edges = G.edges()
    for (u, v), w in zip(edges, weight_norm):
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], 
                'gray', alpha=0.1 + 0.4*w, linewidth=0.5 + 1.5*w)
    
    ax.scatter([pos[i][0] for i in G.nodes()], 
              [pos[i][1] for i in G.nodes()],
              s=50, c='steelblue', alpha=0.7)
    
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')
    ax.set_title(f'Spatial Layout\n(distance_exponent={exp})', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Bottom row: weight distributions
    ax = axes[1, col]
    weights = [d['weight'] for u, v, d in G.edges(data=True)]
    ax.hist(weights, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    ax.set_xlabel('Edge weight', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'Weight Distribution', fontsize=11, fontweight='bold')
    ax.set_yscale('log')

plt.tight_layout()
plt.savefig('bio_inspired_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Biologically inspired graph visualization complete.")

## Comparing Weight Distributions Across Methods

A key feature of biological networks is the **heavy-tailed weight distribution**. Most connections are weak, but a few are very strong. This is often modeled as a **lognormal distribution**.

Let's compare the weight distributions produced by different generation methods.

In [ ]:
# Generate comparable networks using different methods
target_density = 0.1
n = 100

# Inverse-square model
G_inv_sq, _ = generate_bio_inspired_graph(n, distance_exponent=2.0, weight_exponent=1.0, 
                                           density_target=target_density, seed=42)

# Exponential distance model
G_exp, _ = generate_bio_inspired_graph(n, distance_exponent=2.0, weight_exponent=2.0, 
                                        weight_function='exponential', density_target=target_density, seed=42)

# Waxman model
G_wax, _ = generate_waxman_graph(n, alpha=0.3, beta=0.6, seed=42)

# WREN model
G_wren, _ = generate_wren_graph(n, initial_edges=15, time_steps=300, reinforcement=0.08, seed=42)

# Hierarchical modular
G_hier_comp, _ = generate_hierarchical_modular_graph(num_modules=5, nodes_per_module=20, 
                                                       p_intra=0.3, p_inter=0.02, seed=42)

# Extract weights (add 1.0 weight to unweighted models for comparison)
def get_weights(G):
    weights = []
    for u, v, d in G.edges(data=True):
        w = d.get('weight', 1.0)
        weights.append(w)
    return np.array(weights)

weights_inv_sq = get_weights(G_inv_sq)
weights_exp = get_weights(G_exp)
weights_wax = get_weights(G_wax)
weights_wren = get_weights(G_wren)
weights_hier = get_weights(G_hier_comp)

print("Weight statistics across methods:")
methods = [
    ('Inverse-square', weights_inv_sq),
    ('Exponential', weights_exp),
    ('Waxman', weights_wax),
    ('WREN', weights_wren),
    ('Hierarchical', weights_hier),
]

for name, weights in methods:
    print(f"\n{name}:")
    print(f"  Mean: {np.mean(weights):.4f}")
    print(f"  Median: {np.median(weights):.4f}")
    print(f"  Std: {np.std(weights):.4f}")
    print(f"  Skewness (mean/median): {np.mean(weights)/np.median(weights):.2f}x")
    print(f"  Max: {np.max(weights):.4f}")

In [ ]:
# Visualize weight distribution comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

methods_vis = [
    ('Inverse-square', weights_inv_sq, 'steelblue'),
    ('Exponential', weights_exp, 'orange'),
    ('Waxman', weights_wax, 'green'),
    ('WREN', weights_wren, 'red'),
    ('Hierarchical', weights_hier, 'purple'),
]

for idx, (name, weights, color) in enumerate(methods_vis):
    ax = axes[idx]
    ax.hist(weights, bins=30, alpha=0.7, color=color, edgecolor='black')
    ax.set_xlabel('Edge weight', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{name}', fontsize=11, fontweight='bold')
    ax.set_yscale('log')

# Log-log plot for comparison
ax = axes[5]
for name, weights, color in methods_vis:
    # Sort and plot
    sorted_w = np.sort(weights)
    ranks = np.arange(1, len(sorted_w) + 1)
    ax.loglog(sorted_w, ranks, 'o-', label=name, alpha=0.6, markersize=3)

ax.set_xlabel('Edge weight (log)', fontsize=10)
ax.set_ylabel('Rank (log)', fontsize=10)
ax.set_title('Cumulative Distribution Comparison', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('weight_distributions_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Weight distribution comparison visualization complete.")

## Comprehensive Method Comparison Matrix

Let's create a detailed comparison of all methods across key biological properties.

In [ ]:
# Create comprehensive comparison matrix
comparison_data = {
    'Method': [
        'Erdős-Rényi',
        'Watts-Strogatz',
        'Barabási-Albert',
        'Waxman',
        'Spatial Random',
        'Inverse-square',
        'Hierarchical',
        'Stochastic Block',
        'WREN',
    ],
    'Spatial': [0, 0, 0, 2, 2, 2, 0, 0, 1],
    'Weighted': [0, 0, 0, 0, 1, 2, 0, 0, 2],
    'Heterog. Degree': [0, 0, 2, 1, 0, 1, 2, 1, 1],
    'Modularity': [0, 0, 0, 0, 0, 0, 2, 2, 0],
    'Hierarchy': [0, 0, 0, 0, 0, 0, 2, 1, 1],
    'Clustering': [0, 2, 0, 1, 1, 1, 2, 2, 1],
    'Small-world': [0, 2, 1, 1, 1, 2, 1, 1, 1],
    'Sparse': [0, 1, 1, 1, 2, 2, 1, 1, 2],
}

df_comparison = pd.DataFrame(comparison_data)

# Convert to visualization-friendly format
legend_map = {0: '✗', 1: '~', 2: '✓'}
df_display = df_comparison.copy()
for col in df_display.columns:
    if col != 'Method':
        df_display[col] = df_display[col].map(legend_map)

print("\n" + "="*100)
print("COMPREHENSIVE COMPARISON OF GRAPH GENERATION METHODS")
print("="*100)
print(df_display.to_string(index=False))
print("\nLegend: ✗ = Not captured, ~ = Partially captured, ✓ = Well captured")
print("="*100)

In [ ]:
# Visualize comparison as heatmap
fig, ax = plt.subplots(figsize=(12, 8))

# Create heatmap data
heatmap_data = df_comparison.set_index('Method').astype(float)

# Create heatmap
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn', cbar=False, 
            ax=ax, cbar_kws={'label': 'Level of capture'}, linewidths=0.5, 
            vmin=0, vmax=2)

# Customize colorbar appearance with custom labels
cbar = ax.collections[0].colorbar
cbar.set_ticks([0.33, 1, 1.67])
cbar.set_ticklabels(['✗', '~', '✓'])
cbar.set_label('Captures property', fontsize=11)

ax.set_title('Biological Network Properties: Method Comparison Matrix', 
             fontsize=13, fontweight='bold', pad=20)
ax.set_xlabel('Network Properties', fontsize=11, fontweight='bold')
ax.set_ylabel('Graph Generation Method', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('method_comparison_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("Comparison matrix visualization complete.")

In [ ]:
# Compute summary statistics for all methods
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Method implementations
methods_analysis = [
    ('Inverse-square', G_inv_sq),
    ('Exponential', G_exp),
    ('Waxman', G_wax),
    ('WREN', G_wren),
    ('Hierarchical', G_hier_comp),
    ('Stochastic Block', G_sbm),
]

for idx, (name, G) in enumerate(methods_analysis):
    ax = axes[idx // 3, idx % 3]
    
    # Compute metrics
    metrics = {
        'Density': nx.density(G),
        'Avg Clustering': nx.average_clustering(G),
        'Diameter': nx.diameter(G) if nx.is_connected(G) else np.inf,
        'Avg Degree': 2 * G.number_of_edges() / G.number_of_nodes(),
    }
    
    # Normalize for visualization
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())
    
    # Handle infinite values
    metric_values = [v if not np.isinf(v) else 0 for v in metric_values]
    
    colors = ['steelblue', 'coral', 'green', 'red']
    bars = ax.bar(metric_names, metric_values, color=colors, alpha=0.7, edgecolor='black')
    
    ax.set_ylabel('Value', fontsize=10)
    ax.set_title(f'{name}\n({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)', 
                 fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(metric_values) * 1.2)
    
    # Add value labels on bars
    for bar, val in zip(bars, metric_values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('method_metrics_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Method metrics comparison visualization complete.")

## Key Takeaways

### What We've Learned

1. **No single method captures everything**
   - Spatial methods (Waxman, Inverse-square) neglect explicit modularity
   - Modular methods (Hierarchical, SBM) neglect spatial constraints
   - Weight-rich methods (WREN, Inverse-square) lack hierarchy

2. **Spatial + Weighted approaches are most promising**
   - Distance-dependent connection probability provides sparsity
   - Distance-dependent weights provide heterogeneity
   - Natural emergent properties: lognormal weight distributions, small-world topology

3. **Missing elements must be added**
   - **Hierarchy**: Combine with multi-scale construction or use iterative refinement
   - **Modularity**: Add explicit block structure or emergent from spatial clustering
   - **Directionality**: Requires additional asymmetric weights or directed edges

4. **Real biological networks likely use combination mechanisms**
   - Start with spatial + weighted generation (provides backbone)
   - Add modular structure (brain regions, functional areas)
   - Add hierarchy (multiple organizational scales)
   - Prune and refine (remove weak/costly connections)

### The Path Forward: Pruning and Refinement

A critical insight from network neuroscience: **over-connected networks are refined through pruning**. 

Biological development:
1. Generate initial over-connected spatial network
2. Apply pruning rules (activity-dependent, cost-based)
3. Weight refinement (synaptic plasticity)
4. Emergent: sparse, efficient, modular, hierarchical networks

This is the topic of **Notebook 5: Pruning and Network Refinement**.